# Capstone Project 5: Text-to-SQL dengan Verifikasi Eksekusi
Navasena AI, NCA-GENL Batch 6.

**Cara pakai:** jalankan sel secara berurutan dari atas ke bawah (Shift + Enter).
Baca teks di atas setiap sel sebelum menjalankannya. Jika sesi Colab terputus,
jalankan ulang **Bagian 0** lalu lanjutkan dari bagian terakhir.

## Bagian 0. Persiapan (wajib dijalankan setiap membuka sesi baru)

**Sel 0.1** Hubungkan Google Drive. Semua hasil disimpan di Drive agar tidak hilang saat sesi putus. Klik *Connect to Google Drive* lalu izinkan akses.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Google Drive tidak tersedia (Jalur B / VM sendiri). Output disimpan di folder outputs/.", e)

**Sel 0.2** Ambil kode dari GitHub dan install library. Ganti `GANTI_USERNAME_GITHUB` dengan username GitHub Anda. Proses sekitar 2-4 menit.

In [ ]:
import os
GITHUB_USER = "GANTI_USERNAME_GITHUB"
REPO_NAME = "text2sql-capstone"

if os.path.basename(os.getcwd()) != REPO_NAME:
    if not os.path.exists(REPO_NAME):
        !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_NAME}
else:
    !git pull
!pip install -q -r requirements.txt
print("Selesai. Folder kerja:", os.getcwd())

**Sel 0.3** Muat token dari Colab Secrets (ikon kunci di panel kiri). Butuh tiga secret: `HF_TOKEN`, `HF_USERNAME`, `NIM_API_KEY`.

In [ ]:
import os
try:
    from google.colab import userdata
    for key in ["HF_TOKEN", "HF_USERNAME", "NIM_API_KEY"]:
        os.environ[key] = userdata.get(key)
    print("Secrets berhasil dimuat.")
except Exception as e:
    print("Colab Secrets tidak tersedia. Pastikan sudah login lewat terminal VM (Jalur B).", e)

from huggingface_hub import login
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])

**Sel 0.4** Cek GPU dan lokasi penyimpanan. Pastikan nama GPU muncul. Jika muncul error, ubah runtime ke GPU (Runtime > Change runtime type).

In [ ]:
import sys, torch
sys.path.insert(0, ".")
from src import config as C

print("GPU        :", torch.cuda.get_device_name(0))
print("VRAM       :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
print("HF user    :", C.HF_USERNAME)
print("Folder hasil:", C.OUT)

## Bagian 1. Data: Load, Hapus Duplikat, Split per Skema

**Sel 1.1** Unduh dataset, hapus duplikat, lalu bagi menjadi train/val/test berdasarkan skema. Hasil kebocoran harus 0 semua.

In [ ]:
import pandas as pd
from src.data import load_splits, check_leakage, query_type_stats

train_df, val_df, test_df = load_splits()
print("Cek kebocoran skema antar split (harus 0):", check_leakage(train_df, val_df, test_df))

train_df.to_parquet(C.DATA_DIR / "train.parquet")
val_df.to_parquet(C.DATA_DIR / "val.parquet")
test_df.to_parquet(C.DATA_DIR / "test.parquet")

**Sel 1.2** Statistik korpus untuk Section 2 laporan. Perhatikan kolom `join_%`: angkanya kecil, sehingga kemampuan JOIN diuji lewat studi kasus (Bagian 7).

In [ ]:
stats = pd.DataFrame({"train": query_type_stats(train_df),
                      "val": query_type_stats(val_df),
                      "test": query_type_stats(test_df)})
stats.to_csv(C.RESULT_DIR / "statistik_korpus.csv")
stats

## Bagian 2. Uji Evaluator

**Sel 2.1** Pastikan evaluator bekerja benar sebelum dipakai. Output yang diharapkan: baris 1 `ex=True`, baris 2 `ex=False`, baris 3 dan 4 `syntax_ok=False`.

In [ ]:
from src.execution_evaluator import evaluate_pair

ctx = "CREATE TABLE head (age INTEGER, name VARCHAR)"
gold = "SELECT COUNT(*) FROM head WHERE age > 56"
print("1. Prediksi identik    :", evaluate_pair(ctx, gold, gold))
print("2. Prediksi salah      :", evaluate_pair(ctx, gold, "SELECT COUNT(*) FROM head WHERE age > 10"))
print("3. Prediksi rusak      :", evaluate_pair(ctx, gold, "SELEC nonsense"))
print("4. Prediksi berbahaya  :", evaluate_pair(ctx, gold, "DELETE FROM head"))

## Bagian 3. Baseline (sebelum fine-tuning)

**Sel 3.1** Ambil 1.000 sampel test tetap (seed 42). Sampel yang sama dipakai untuk semua model agar perbandingan adil.

In [ ]:
from src.prompt import build_prompt

test_eval = test_df.sample(C.N_TEST_EVAL, random_state=C.SEED).reset_index(drop=True)
test_eval.to_parquet(C.DATA_DIR / "test_eval.parquet")
prompts = [build_prompt(r.context, r.question) for r in test_eval.itertuples()]
print(len(prompts), "prompt siap. Contoh:")
print(prompts[0][1]["content"])

**Sel 3.2** Generate SQL dengan model dasar Qwen2.5-1.5B (tanpa fine-tuning). Estimasi 5-10 menit di T4.

In [ ]:
from src.infer import load_hf_model, generate_hf
from src.execution_evaluator import evaluate_frame, summarize

model, tok = load_hf_model(C.BASE_MODEL)
test_eval["pred_base"] = generate_hf(model, tok, prompts)
res_base = evaluate_frame(test_eval, "pred_base")
res_base.to_csv(C.RESULT_DIR / "hasil_base.csv", index=False)
summarize(res_base)

**Sel 3.3** Bebaskan memori GPU sebelum lanjut.

In [ ]:
import gc
del model; gc.collect(); torch.cuda.empty_cache()
print("VRAM terpakai:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

**Sel 3.4** Baseline model raksasa Llama-3.3-70B via NVIDIA NIM pada 200 sampel pertama. Estimasi 5-10 menit karena ada jeda antar request.

In [ ]:
from src.infer import generate_nim

nim_df = test_eval.head(C.N_NIM_EVAL).copy()
nim_df["pred_nim"] = generate_nim(prompts[: C.N_NIM_EVAL], C.NIM_MODEL)
res_nim = evaluate_frame(nim_df, "pred_nim")
res_nim.to_csv(C.RESULT_DIR / "hasil_nim.csv", index=False)
summarize(res_nim)

## Bagian 4. Fine-Tuning QLoRA

**Sel 4.1 (KHUSUS BUKTI T4)** Jalankan sel ini **hanya** saat memakai Colab gratis dengan GPU T4.
Tujuannya membuktikan pelatihan muat di T4 (Prinsip Keras 1). Cukup 50 langkah. Catat angka VRAM puncak untuk laporan.
Jika sedang memakai GPU besar, lewati sel ini.

In [ ]:
from src.train import train_qlora

torch.cuda.reset_peak_memory_stats()
_ = train_qlora(train_df, val_df, output_dir=C.CKPT_DIR / "uji_t4", max_steps=50)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM puncak:", round(torch.cuda.max_memory_allocated() / 1e9, 2), "GB")

**Sel 4.2** Pelatihan penuh (1 epoch, 15.000 sampel). Estimasi 40-70 menit di T4, lebih cepat di GPU besar.
Checkpoint disimpan otomatis setiap 200 langkah. Jika sesi putus, jalankan Bagian 0, Sel 1.1, lalu sel ini lagi. Pelatihan otomatis dilanjutkan dari checkpoint terakhir.

In [ ]:
from src.train import train_qlora

trainer = train_qlora(train_df, val_df, output_dir=C.CKPT_DIR / "full")
trainer.save_model(str(C.CKPT_DIR / "adapter_final"))
print("Adapter tersimpan di", C.CKPT_DIR / "adapter_final")

**Sel 4.3** Simpan grafik training loss dan validation loss untuk Section 4 laporan.

In [ ]:
import matplotlib.pyplot as plt

log = pd.DataFrame(trainer.state.log_history)
tr_loss = log.dropna(subset=["loss"])
ev_loss = log.dropna(subset=["eval_loss"])
plt.figure(figsize=(7, 4))
plt.plot(tr_loss["step"], tr_loss["loss"], label="Training loss")
plt.plot(ev_loss["step"], ev_loss["eval_loss"], marker="o", label="Validation loss")
plt.xlabel("Step"); plt.ylabel("Loss"); plt.legend(); plt.title("Kurva Loss QLoRA")
plt.savefig(C.RESULT_DIR / "kurva_loss.png", dpi=150, bbox_inches="tight")
plt.show()

**Sel 4.4** Unggah adapter ke Hugging Face Hub.

In [ ]:
trainer.model.push_to_hub(C.REPO_ADAPTER)
trainer.processing_class.push_to_hub(C.REPO_ADAPTER)
print("Adapter diunggah ke https://huggingface.co/" + C.REPO_ADAPTER)
del trainer; gc.collect(); torch.cuda.empty_cache()

## Bagian 5. Evaluasi Model Fine-Tuned

**Sel 5.1** Generate SQL dengan model hasil fine-tuning pada 1.000 sampel test yang sama.

In [ ]:
model, tok = load_hf_model(C.BASE_MODEL, adapter=C.REPO_ADAPTER)
test_eval["pred_ft"] = generate_hf(model, tok, prompts)
res_ft = evaluate_frame(test_eval, "pred_ft")
res_ft.to_csv(C.RESULT_DIR / "hasil_finetuned.csv", index=False)
summarize(res_ft)

## Bagian 6. Tabel Perbandingan, Uji Signifikansi, Analisis Galat

**Sel 6.0 (opsional)** Jalankan hanya jika sesi sempat putus setelah Bagian 5. Sel ini memuat ulang hasil evaluasi dari file, jadi Anda tidak perlu generate ulang.

In [ ]:
from src.execution_evaluator import summarize
from src.prompt import build_prompt
import gc
res_base = pd.read_csv(C.RESULT_DIR / "hasil_base.csv")
res_ft = pd.read_csv(C.RESULT_DIR / "hasil_finetuned.csv")
res_nim = pd.read_csv(C.RESULT_DIR / "hasil_nim.csv")
test_eval = pd.read_parquet(C.DATA_DIR / "test_eval.parquet")
prompts = [build_prompt(r.context, r.question) for r in test_eval.itertuples()]
print("Hasil dimuat ulang.")

**Sel 6.1** Tabel utama untuk Section 4 laporan.

In [ ]:
from src.stats_utils import mcnemar_test, bootstrap_ci

ids_nim = set(range(C.N_NIM_EVAL))
tabel = pd.DataFrame({
    "Base Qwen 1.5B (n=1000)": summarize(res_base),
    "Fine-tuned Qwen 1.5B (n=1000)": summarize(res_ft),
    "Base Qwen 1.5B (n=200)": summarize(res_base.head(C.N_NIM_EVAL)),
    "Fine-tuned Qwen 1.5B (n=200)": summarize(res_ft.head(C.N_NIM_EVAL)),
    "Llama-3.3-70B NIM (n=200)": summarize(res_nim),
}).T
tabel.to_csv(C.RESULT_DIR / "tabel_perbandingan.csv")
tabel

**Sel 6.2** Uji McNemar (base vs fine-tuned) dan confidence interval 95%. Kenaikan signifikan jika p-value < 0,05.

In [ ]:
mask = res_base["gold_ok"] & res_ft["gold_ok"]
uji = mcnemar_test(res_base.loc[mask, "ex"], res_ft.loc[mask, "ex"])
uji["CI95_EX_base_%"] = bootstrap_ci(res_base.loc[mask, "ex"])
uji["CI95_EX_finetuned_%"] = bootstrap_ci(res_ft.loc[mask, "ex"])
pd.Series(uji).to_csv(C.RESULT_DIR / "uji_signifikansi.csv")
pd.Series(uji)

**Sel 6.3** Skor per jenis query (JOIN vs non-JOIN) dan kategori galat otomatis.

In [ ]:
from src.data import has_join
from src.stats_utils import auto_tag

res_ft["tipe"] = res_ft["answer"].map(lambda s: "join" if has_join(s) else "non_join")
per_tipe = res_ft[res_ft["gold_ok"]].groupby("tipe")[["syntax_ok", "em", "ex"]].mean().mul(100).round(2)
per_tipe["n"] = res_ft[res_ft["gold_ok"]].groupby("tipe").size()
display(per_tipe)

salah = res_ft[res_ft["gold_ok"] & ~res_ft["ex"]].copy()
salah["kategori_otomatis"] = [auto_tag(g, p, s) for g, p, s in zip(salah["answer"], salah["pred"], salah["syntax_ok"])]
display(salah["kategori_otomatis"].value_counts())

sampel = salah.sample(min(50, len(salah)), random_state=C.SEED)
sampel["kategori_manual"] = ""
sampel[["question", "context", "answer", "pred", "kategori_otomatis", "kategori_manual"]].to_csv(
    C.RESULT_DIR / "sampel_galat_untuk_dicek.csv", index=False)
print("File sampel_galat_untuk_dicek.csv siap diperiksa manual.")

## Bagian 7. Studi Kasus E-Commerce Indonesia (uji JOIN dan bahasa Indonesia)

**Sel 7.1** Uji model fine-tuned pada 20 pertanyaan berbahasa Indonesia terhadap database e-commerce.

In [ ]:
import json, sqlite3, gc
from src.infer import load_hf_model, generate_hf
from src.execution_evaluator import evaluate_on_db

cs = pd.read_json(C.CASE_STUDY_FILE, lines=True)
con = sqlite3.connect(C.ECOMMERCE_DB)
schema_cs = "\n".join(r[0] for r in con.execute("SELECT sql FROM sqlite_master WHERE type='table'"))
con.close()
cs_prompts = [build_prompt(schema_cs, q) for q in cs["question"]]

def eval_case_study(preds):
    rows = [evaluate_on_db(C.ECOMMERCE_DB, g, p) for g, p in zip(cs["gold"], preds)]
    return pd.DataFrame(rows)

if "model" in globals():
    del model; gc.collect(); torch.cuda.empty_cache()
model, tok = load_hf_model(C.BASE_MODEL, adapter=C.REPO_ADAPTER)
cs["pred_ft"] = generate_hf(model, tok, cs_prompts)
cs_ft = pd.concat([cs, eval_case_study(cs["pred_ft"])], axis=1)
cs_ft[["question", "pred_ft", "ex"]]

**Sel 7.2** Uji model dasar pada studi kasus yang sama, lalu ringkas hasilnya.

In [ ]:
del model; gc.collect(); torch.cuda.empty_cache()
model, tok = load_hf_model(C.BASE_MODEL)
cs["pred_base"] = generate_hf(model, tok, cs_prompts)
cs_base = pd.concat([cs, eval_case_study(cs["pred_base"])], axis=1)

ringkas_cs = pd.DataFrame({
    "base": cs_base.groupby("tipe")["ex"].mean().mul(100).round(1),
    "fine_tuned": cs_ft.groupby("tipe")["ex"].mean().mul(100).round(1),
})
ringkas_cs.loc["total"] = [cs_base["ex"].mean() * 100, cs_ft["ex"].mean() * 100]
cs_ft.to_csv(C.RESULT_DIR / "studi_kasus_finetuned.csv", index=False)
cs_base.to_csv(C.RESULT_DIR / "studi_kasus_base.csv", index=False)
ringkas_cs.to_csv(C.RESULT_DIR / "studi_kasus_ringkas.csv")
del model; gc.collect(); torch.cuda.empty_cache()
ringkas_cs

## Bagian 8. Merge, Konversi GGUF, Unggah ke Hugging Face

**Sel 8.1** Gabungkan adapter ke bobot model fp16. Estimasi 2-3 menit.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(C.BASE_MODEL, torch_dtype=torch.float16)
merged = PeftModel.from_pretrained(base, C.REPO_ADAPTER).merge_and_unload()
merged.save_pretrained("merged")
AutoTokenizer.from_pretrained(C.BASE_MODEL).save_pretrained("merged")
del base, merged; gc.collect()
print("Model gabungan tersimpan di folder merged/")

**Sel 8.2** Konversi ke GGUF dan kuantisasi Q4_K_M. Estimasi 5-10 menit (kompilasi llama.cpp).

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q ./llama.cpp/gguf-py sentencepiece protobuf
!python llama.cpp/convert_hf_to_gguf.py merged --outfile model-f16.gguf --outtype f16
!cmake -S llama.cpp -B build -DLLAMA_CURL=OFF > /dev/null && cmake --build build --target llama-quantize -j 4 > /dev/null
!./build/bin/llama-quantize model-f16.gguf {C.GGUF_FILE} Q4_K_M
!ls -lh *.gguf

**Sel 8.3** Unggah file GGUF dan Modelfile Ollama ke Hugging Face Hub (repo publik).

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(C.REPO_GGUF, exist_ok=True, private=False)
api.upload_file(path_or_fileobj=C.GGUF_FILE, path_in_repo=C.GGUF_FILE, repo_id=C.REPO_GGUF)
api.upload_file(path_or_fileobj="ollama/Modelfile", path_in_repo="Modelfile", repo_id=C.REPO_GGUF)
print("GGUF diunggah ke https://huggingface.co/" + C.REPO_GGUF)

## Bagian 9. Evaluasi Ulang Model GGUF (dampak kuantisasi)

**Sel 9.1** Install llama-cpp-python lalu uji GGUF pada 150 sampel test. Berjalan di CPU, estimasi 15-30 menit.

In [ ]:
!pip install -q llama-cpp-python==0.3.2 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
from src.infer import generate_gguf

gguf_df = test_eval.head(C.N_GGUF_EVAL).copy()
gguf_df["pred_gguf"] = generate_gguf(prompts[: C.N_GGUF_EVAL], C.REPO_GGUF, C.GGUF_FILE)
res_gguf = evaluate_frame(gguf_df, "pred_gguf")
res_gguf.to_csv(C.RESULT_DIR / "hasil_gguf.csv", index=False)

pd.DataFrame({
    "fp16 + adapter (n=150)": summarize(res_ft.head(C.N_GGUF_EVAL)),
    "GGUF Q4_K_M (n=150)": summarize(res_gguf),
}).T

## Bagian 10. Selesai

**Sel 10.1** Daftar semua file hasil. File-file ini dipakai untuk laporan.

In [ ]:
for f in sorted(C.RESULT_DIR.iterdir()):
    print(f.name)